# Using `transformers` Models via Pipelines

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/Building_with_Deep_Learning/01-llms/02_pipelines.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

_Click the badge above to open and run this notebook in Google Colab!_

In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/Building_with_Deep_Learning/01-llms"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Overview

The [`transformers`](https://huggingface.co/docs/transformers/index) library provides models that are faithful to their papers, easy to use, and easy to hack.

- **Engineers** who want a pretrained model that “just works” with a predictable API.
- **Practitioners** fine-tuning, evaluating, or serving models.
- Researchers and educators exploring or extending model architectures (PyTorch-first).

## Set up

To start, we recommend creating a Hugging Face [account](https://hf.co/join). An account lets you host and access version controlled models, datasets, and [Spaces](https://hf.co/spaces) on the Hugging Face [Hub](https://hf.co/docs/hub/index), a collaborative platform for discovery and building.

Create a [User Access Token](https://hf.co/docs/hub/security-tokens#user-access-tokens) and log in to your account.

### Install PyTorch



In [1]:
!pip install -qqq torch

### Update Libraries

Then install an **up-to-date version of Transformers and some additional libraries** from the Hugging Face ecosystem for accessing datasets and vision models, evaluating training, and optimizing training for large models.

In [2]:
!pip install -qqqU transformers datasets evaluate accelerate timm

### Suppress output logs

In [ ]:
import os
import logging

from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
disable_progress_bars()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

## Accelerator using `accelerate` library

[Accelerate](https://hf.co/docs/accelerate/index) is a library of distributed training. It loads and stores the model weights on the **fastest device first (GPU)**, and then moves the weights to other devices (CPU, hard drive) as needed.

In [4]:
from accelerate import Accelerator

device = Accelerator().device

## Pipeline

The [Pipeline](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/pipelines#transformers.Pipeline) class is the most convenient way to inference with a pretrained model. It supports many tasks such as text generation, image segmentation, automatic speech recognition, document question answering, and more.

> Refer to the [Pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines) API reference for a complete list of available tasks.

Create a [Pipeline](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/pipelines#transformers.Pipeline) object and select a task. By default, [Pipeline](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/pipelines#transformers.Pipeline) downloads and caches a default pretrained model for a given task. Pass the model name to the `model` parameter to choose a specific model.

### Example 1: Speech Recognition

In [5]:
from transformers import pipeline

asr = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3",
    device=device
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Pass an audio file to [Pipeline](https://huggingface.co/docs/transformers/v5.12.0/en/main_classes/pipelines#transformers.Pipeline).

In [6]:
asr_output = asr("https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac")
asr_output

{'text': ' He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick, peppered, flour-fattened sauce.'}

### Example 2: Zero-shot Classification

The following example shows a pipeline for [*Zero-shot Classification*](https://huggingface.co/models?pipeline_tag=zero-shot-classification&sort=trending):

In [ ]:
from transformers import pipeline

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device_map="auto" # Uses accelerate
)

In [13]:
sequence_to_classify = "I've been waiting for a HuggingFace course my whole life."
candidate_labels = ['machine learning', 'algebra', 'history']

output = classifier(sequence_to_classify, candidate_labels)
output

{'sequence': "I've been waiting for a HuggingFace course my whole life.",
 'labels': ['machine learning', 'history', 'algebra'],
 'scores': [0.4895123541355133, 0.3653002977371216, 0.14518731832504272]}

### Example 3: OpenAI Privacy Filter

In [14]:
from transformers import pipeline

token_classifier = pipeline(
    task="token-classification",
    model="openai/privacy-filter",
)

In [17]:
output = token_classifier("My name is Alice Smith and my address is 123 Main St, Anytown, USA")
output

[{'entity': 'B-private_person',
  'score': np.float32(0.9999919),
  'index': 3,
  'word': 'ĠAlice',
  'start': 10,
  'end': 16},
 {'entity': 'E-private_person',
  'score': np.float32(0.99999857),
  'index': 4,
  'word': 'ĠSmith',
  'start': 16,
  'end': 22},
 {'entity': 'B-private_address',
  'score': np.float32(0.99995816),
  'index': 10,
  'word': '123',
  'start': 41,
  'end': 44},
 {'entity': 'I-private_address',
  'score': np.float32(0.99998367),
  'index': 11,
  'word': 'ĠMain',
  'start': 44,
  'end': 49},
 {'entity': 'I-private_address',
  'score': np.float32(0.99999654),
  'index': 12,
  'word': 'ĠSt',
  'start': 49,
  'end': 52},
 {'entity': 'I-private_address',
  'score': np.float32(0.9999974),
  'index': 13,
  'word': ',',
  'start': 52,
  'end': 53},
 {'entity': 'I-private_address',
  'score': np.float32(0.9999981),
  'index': 14,
  'word': 'ĠAn',
  'start': 53,
  'end': 56},
 {'entity': 'I-private_address',
  'score': np.float32(0.9999976),
  'index': 15,
  'word': 'yt',


## Available pipelines for different modalities

The `pipeline()` function supports multiple modalities, allowing you to work with text, images, audio, and even multimodal tasks. In this course we’ll focus on text tasks, but it’s useful to understand the transformer architecture’s potential, so we’ll briefly outline it.

| Task | Modality | Output | Models | Datasets |
|---|---|---|---|---|
| [`audio-classification`](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.AudioClassificationPipeline) | Audio | Classify audio into categories | [🔗](https://huggingface.co/models?filter=audio-classification) | [🔗](https://huggingface.co/datasets?task_categories=audio-classification) |
| [`image-classification`](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ImageClassificationPipeline) | Image | Predicted class for an image | [🔗](https://huggingface.co/models?filter=image-classification) | [🔗](https://huggingface.co/datasets?task_categories=image-classification) |
| [`object-detection`](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ObjectDetectionPipeline) | Image | Locate and identify objects in images | [🔗](https://huggingface.co/models?filter=object-detection) | [🔗](https://huggingface.co/datasets?task_categories=object-detection) |

Refer to the [Pipelines](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines) API reference for a complete list of available tasks. 


## Joining Pipelines

Here's an [interactive demo of a sequence-to-sequence model for translation](https://course-demos-speech-to-speech-translation.hf.space).


## Batch Inference

Batch inference is disabled by default since  hardware, data, and the model itself can affect whether it improves speed or not. In the example below, when there are 4 inputs and `batch_size=2` , Pipeline passes a batch of 2 inputs, **twice**.

In [18]:
from transformers import pipeline

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device_map="auto", # Uses accelerate
    batch_size=2,

)

In [20]:
classifier([
    "I am very happy",
    "The product was great",
    "I'm very sad",
],
    candidate_labels=["positive review", "negative review"]
)

[{'sequence': 'I am very happy',
  'labels': ['positive review', 'negative review'],
  'scores': [0.9979938268661499, 0.0020061207469552755]},
 {'sequence': 'The product was great',
  'labels': ['positive review', 'negative review'],
  'scores': [0.9963841438293457, 0.0036158019211143255]},
 {'sequence': "I'm very sad",
  'labels': ['negative review', 'positive review'],
  'scores': [0.9949876666069031, 0.005012379493564367]}]

## Batch Inference: rules of thumb

- If you are latency constrained (live product doing inference), don’t batch.
- The larger the GPU the more likely batching is going to be more interesting.
- If you are using CPU, don’t batch.
- Handle OOM errors.

Read more at: [Pipeline batching](https://huggingface.co/docs/transformers/main_classes/pipelines#pipeline-batching).
